# Deploy a Custom JupyterHub & Build Images in NRP GitLab

**NAIRR AI Education Webinar Series** · [website version](https://training.nrp-nautilus.io/nairr-webinar/2_custom_jupyterhub.html) — run cells with **Shift+Enter**.


> ### ⚠️ Read this before you deploy a *real* hub
>
> This tutorial takes a deliberate shortcut so it fits in a webinar. Every hub you deploy **after** today should follow the documented path in [Deploy JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/), and the difference that matters is **authentication**.
>
> | | This tutorial | A hub you actually run |
> |---|---|---|
> | Authenticator | `DummyAuthenticator` | `CILogonOAuthenticator` |
> | Who can sign in | anyone who knows the shared password | your campus IdP, narrowed by `allowed_idps` / `allowed_users` |
> | Prerequisite | none | an OAuth client registered with CILogon |
> | Lead time | zero | **plan on several days to more than a week** |
>
> `DummyAuthenticator` is a password in a values file — fine for a throwaway namespace for one hour, **not** acceptable for a hub on a public hostname. NRP's docs are blunt about this: leaving a hub open for anyone to sign in can get your namespace locked.
>
> The real path uses **CILogon**, the same federated login NRP itself uses, so students sign in with existing campus credentials. The catch: CILogon is an **independent service, not operated by NRP**, and you register your own OAuth client with them at [cilogon.org/oauth2/register](https://cilogon.org/oauth2/register) — callback `https://<your-hostname>.nrp-nautilus.io/hub/oauth_callback`, client type *Confidential*, refresh tokens *No*, scopes `org.cilogon.userinfo,openid,profile,email`.
>
> CILogon staff review each registration by hand and email you a client ID and secret once approved — **budget a few days, and it can stretch past a week**. So: **start the registration well before the term** (nothing on the NRP side unblocks it), and **pick your hostname first**, since it is baked into the callback URL you register.
>
> Everything else here — Helm, the values file, profiles, resource limits, shared storage, custom images — is identical either way. Only the `hub.config` authentication block changes, plus an [`allowed_idps` allowlist](https://cilogon.org/idplist/) for your institution.


> ### 🧰 Tools you need on your own machine
>
> This training hub has all of it preinstalled, so nothing below is needed *today*. To run the same commands from your laptop against your own namespace, you need three tools and the cluster config:
>
> | What | Why | Where |
> |---|---|---|
> | `kubectl` | talks to the Kubernetes API — every `kubectl` command in this notebook | [kubernetes.io/docs/tasks/tools](https://kubernetes.io/docs/tasks/tools/) |
> | **`kubelogin`** | CILogon/OIDC login for `kubectl`. **The NRP kubeconfig does not work without it** | [github.com/int128/kubelogin](https://github.com/int128/kubelogin) |
> | `helm` | installs and upgrades the JupyterHub chart | [helm.sh/docs/intro/install](https://helm.sh/docs/intro/install/) |
> | **Nautilus kubeconfig** | points `kubectl` at Nautilus and carries your CILogon identity — save it as `~/.kube/config`, no extension | [nrp.ai/config](https://nrp.ai/config) |
>
> `kubelogin` is a `kubectl` plugin, so the binary must land on your `PATH` under the name **`kubectl-oidc_login`** — that exact name is how `kubectl` finds it. The NRP docs give a copy-paste installer for Linux and macOS, plus fixes for headless machines, WSL, and port conflicts: [cluster access via `kubectl`](https://nrp.ai/documentation/userdocs/start/getting-started/#cluster-access-via-kubectl).
>
> Fetch the kubeconfig and confirm the whole chain works:
>
> ```bash
> mkdir -p ~/.kube
> curl -o ~/.kube/config -fSL https://nrp.ai/config
>
> kubectl config get-contexts     # should list the `nautilus` context
> kubectl auth whoami             # triggers the CILogon browser login the first time
> helm version --short
> ```
>
> You also need to be an **admin** of the namespace you deploy into — a plain member cannot install a chart.


## ⚙️ Setup — run this first

Set your short username once; every command below uses `$NRP_USER`. The cell
also renders every manifest into **`my-yamls/`** with `<username>` already
filled in — wherever the website says *"replace `<username>`"*, it's already
done for you here.

> Terminal steps below don't share this variable — run the same
> `export NRP_USER=...` line in any terminal you open.
> Re-running this cell re-renders `my-yamls/` (overwriting any edits you made there).

**First time in one of these notebooks?** Click the **📌 pin icon** in the toolbar
above for a 30-second guided tour of how this notebook works.


In [ ]:
export NRP_USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/nairr-webinar/workspace
if [ "$NRP_USER" = changeme ]; then echo "⚠️  Edit NRP_USER above first, then re-run"; else
  mkdir -p my-yamls
  for f in yamls/*; do sed "s/<username>/$NRP_USER/g" "$f" > "my-yamls/$(basename "$f")"; done
  echo "✅ my-yamls/ rendered for $NRP_USER"
fi
# claim your own namespace for the session (idempotent — same slot every time you ask):
export NRP_NAMESPACE=$(curl -s "http://nrp-claim.nrp-training.svc.cluster.local/claim?user=${JUPYTERHUB_USER:-$NRP_USER}")
export NRP_RELEASE=jhub-$NRP_USER
echo "namespace=$NRP_NAMESPACE release=$NRP_RELEASE"


Deploy your **own** JupyterHub with Helm — controlled access, custom images, per-profile resource limits, shared storage — then see how to build custom container images with NRP GitLab CI/CD. This is the recipe instructors and PIs use to stand up course and lab hubs on NRP.

**Conventions.** Each participant works in their **own pre-created namespace** (`nrp-training-000` … `nrp-training-099`) — JupyterHub can only be deployed once per namespace. Claim yours now; the request is keyed by your hub login, so it's idempotent — you get the **same** slot back every time, and re-running this cell after a break is safe:


In [ ]:
export NRP_NAMESPACE=$(curl -s "http://nrp-claim.nrp-training.svc.cluster.local/claim?user=${JUPYTERHUB_USER:-$NRP_USER}")
export NRP_RELEASE=jhub-$NRP_USER
echo "namespace=$NRP_NAMESPACE release=$NRP_RELEASE"


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
namespace=nrp-training-042 release=jhub-alice
</pre>
</details>


`$NRP_NAMESPACE` and `$NRP_RELEASE` are what the commands below (and `check.sh 2`) pick up — no hand-editing. Replace `<namespace>`/`<release-name>` in any manifest with these.

> 📘 **Docs:** [Deploy JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/) · [Build images](https://nrp.ai/documentation/userdocs/tutorial/images/) · [NRP GitLab CI](https://nrp.ai/documentation/userdocs/development/gitlab/) · [Z2JH (upstream)](https://z2jh.jupyter.org)


## 1. Helm in one paragraph

Helm is a package manager for Kubernetes — instead of authoring every Deployment, Service, and ConfigMap by hand, you install a **chart** (a reusable bundle of templates) and tune it through a **values file**. The [Zero to JupyterHub chart](https://z2jh.jupyter.org) packages the entire hub/proxy/spawner stack; your whole deployment is one YAML file of values.

In the tutorial hub, `helm` is preinstalled — verify, then add the chart repository:


In [ ]:
kubectl auth whoami && helm version --short

helm repo add jupyterhub https://jupyterhub.github.io/helm-chart/
helm repo update
helm repo list


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
&quot;jupyterhub&quot; has been added to your repositories
Update Complete. ⎈Happy Helming!⎈

NAME         URL
jupyterhub   https://jupyterhub.github.io/helm-chart/
</pre>
</details>


## 2. Examine the values file

Open `yamls/jhub-values.yaml`. Key sections:

```yaml
hub:
  config:
    JupyterHub:
      authenticator_class: dummy      # tutorial only — swap for CILogon/OIDC in production
      admin_access: true
      admin_users: ["admin"]
    DummyAuthenticator:
      password: "training123"
  db:
    type: sqlite-pvc
    pvc:
      accessModes: [ReadWriteOnce]
      storage: 1Gi
      storageClassName: rook-ceph-block-east
proxy:
  secretToken: 'secret_token'         # replace before deploying!
singleuser:
  storage:
    type: dynamic
    capacity: 5Gi
    homeMountPath: /home/jovyan
    dynamic:
      storageClass: rook-ceph-block-east
      pvcNameTemplate: claim-{username}{servername}
      storageAccessModes: [ReadWriteOnce]
  image:
    name: quay.io/jupyter/scipy-notebook
    tag: 2024-04-22
  cpu: {limit: 2, guarantee: 2}
  memory: {limit: 8G, guarantee: 8G}
  defaultUrl: "/lab"
cull:                                  # required on NRP — close inactive sessions
  enabled: true
  timeout: 3600
  every: 600
```


Generate a real proxy token and put it in the file in place of `secret_token`:


In [ ]:
openssl rand -hex 32


## 3. Deploy

### First — what's already running in your namespace?

JupyterHub can only be deployed **once per namespace**: a second release fights
the first one over the `proxy-public` service and the hub database. Your claimed
slot should be empty, but check before you install — if you've run this tutorial
before, or the slot was recycled, there may already be a hub sitting in it.


In [ ]:
helm list -n $NRP_NAMESPACE
kubectl get pods -n $NRP_NAMESPACE


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
NAME	NAMESPACE	REVISION	STATUS	CHART	APP VERSION
No resources found in nrp-training-042 namespace.
</pre>
</details>


An empty `helm list` and no pods means you're clear — **skip the next cell** and
deploy.

If a release *is* listed, look at the `NAME` column. Tear it down only if it's
yours; in a shared namespace someone else's class may be running on it. Set
`OLD_RELEASE` to that name and run the cell below — otherwise leave it alone and
ask whoever owns the namespace.


In [ ]:
# ⚠️  Optional — only if the cell above listed a JupyterHub you want gone.
OLD_RELEASE=changeme   # ✏️ EDIT to the NAME shown by `helm list` above

if [ "$OLD_RELEASE" = changeme ]; then
  echo "Nothing to do — edit OLD_RELEASE above only if you need to remove an existing hub."
else
  helm uninstall "$OLD_RELEASE" -n $NRP_NAMESPACE
  # wait for the pods to actually go away before redeploying
  kubectl wait --for=delete pod -l app=jupyterhub -n $NRP_NAMESPACE --timeout=120s 2>/dev/null || true
  helm list -n $NRP_NAMESPACE
  kubectl get pods -n $NRP_NAMESPACE
fi


`helm uninstall` leaves PVCs behind on purpose — the hub database and any user
home directories survive, so a reinstall picks them back up. Section 8 shows how
to delete those too if you really want a clean slate.

### Install the chart


In [ ]:
helm upgrade --cleanup-on-fail --install $NRP_RELEASE jupyterhub/jupyterhub \
  --namespace $NRP_NAMESPACE \
  --values my-yamls/jhub-values.yaml \
  --wait \
  --timeout=10m


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
Release &quot;jhub-alice&quot; does not exist. Installing it now.
NAME: jhub-alice
NAMESPACE: nrp-training-042
STATUS: deployed
REVISION: 1
NOTES:
       You have successfully installed the official JupyterHub Helm chart!
</pre>
</details>


Inspect what the chart created — every one of these is an ordinary Kubernetes object:


In [ ]:
kubectl get pods -n $NRP_NAMESPACE


In [ ]:
kubectl get services -n $NRP_NAMESPACE


In [ ]:
kubectl get pvc -n $NRP_NAMESPACE


You should see the **hub** pod (auth, sessions, spawning), the **proxy** pod (routing), a `hub-db-dir` PVC — and, once someone logs in, per-user pods and `claim-<user>` PVCs.


## 4. Expose it with an Ingress

`my-yamls/jhub-values.yaml` already has an ingress block at the bottom, commented out, with **your** hostname rendered in — the setup cell substituted `<username>` for you, so it is globally unique. Uncomment it:

```yaml
ingress:
  enabled: true
  ingressClassName: haproxy
  hosts: ["jhub-<username>.nrp-nautilus.io"]
  pathSuffix: ''
  tls:
    - hosts:
      - jhub-<username>.nrp-nautilus.io
```

The quickest way is a one-liner that strips the leading `#` from those lines:


In [ ]:
sed -i '/^#ingress:/,$ s/^#//' my-yamls/jhub-values.yaml
tail -9 my-yamls/jhub-values.yaml


In [ ]:
helm upgrade $NRP_RELEASE jupyterhub/jupyterhub \
  --namespace $NRP_NAMESPACE \
  --values my-yamls/jhub-values.yaml \
  --wait --timeout=10m


In [ ]:
kubectl get ingress -n $NRP_NAMESPACE


After ~a minute for HAProxy + Let's Encrypt, open `https://jhub-$NRP_USER.nrp-nautilus.io`, log in as `admin` with the Dummy password, and spawn a server. **You now have a working multi-user JupyterHub on national research infrastructure.**


## 5. Make it yours


### 5.1 Multiple image profiles

Give users a menu of environments — add to `singleuser`:

```yaml
singleuser:
  profileList:
  - display_name: Scipy
    kubespawner_override:
      image_spec: quay.io/jupyter/scipy-notebook:2024-04-22
    default: True
  - display_name: Tensorflow (CUDA)
    kubespawner_override:
      image_spec: quay.io/jupyter/tensorflow-notebook:cuda-2024-04-22
  - display_name: Pytorch (CUDA 12)
    kubespawner_override:
      image_spec: quay.io/jupyter/pytorch-notebook:cuda12-2024-04-22
  - display_name: Datascience (scipy, Julia, R)
    kubespawner_override:
      image_spec: quay.io/jupyter/datascience-notebook:2024-04-22
```


### 5.2 Per-profile resource limits

```yaml
  - display_name: Small (2 CPU, 4GB RAM)
    kubespawner_override:
      image_spec: quay.io/jupyter/scipy-notebook:2024-04-22
      cpu_limit: 2
      cpu_guarantee: 2
      mem_limit: 4G
      mem_guarantee: 4G
  - display_name: Large (8 CPU, 16GB RAM)
    kubespawner_override:
      image_spec: quay.io/jupyter/scipy-notebook:2024-04-22
      cpu_limit: 8
      cpu_guarantee: 8
      mem_limit: 16G
      mem_guarantee: 16G
```


Add a profile or two to your values file, `helm upgrade` again, and reload the spawn page — the menu updates live. A GPU profile adds `extra_resource_limits: {"nvidia.com/gpu": "1"}`.

This is where a course gets shaped: an intro unit gets a Small CPU profile, a deep-learning unit gets a GPU profile, and students pick from a dropdown instead of you managing machines.


### 5.3 Shared storage for the whole class

Mount the RWX CephFS volume from the storage episode into **every** user server:

```yaml
singleuser:
  storage:
    extraVolumes:
      - name: jupyterhub-shared
        persistentVolumeClaim:
          claimName: jupyterhub-shared-volume
    extraVolumeMounts:
      - name: jupyterhub-shared
        mountPath: /home/shared
```


Instructors drop datasets and notebooks into `/home/shared` once; every student sees them instantly.


### 5.4 Real authentication

For production, replace the Dummy authenticator with institutional login. `yamls/cilogon-jupyterhub-config.yaml` in the workspace shows a CILogon/OIDC configuration — campus credentials, an allowlist or admin-managed access, no passwords to distribute.

Swapping it in needs a `client_id` and `client_secret` from CILogon, which you have to request from them and wait on — see the lead-time warning at the top of this notebook. That wait is the reason today's hub uses the Dummy authenticator.


## 6. Operating your hub


In [ ]:
helm list -n $NRP_NAMESPACE


In [ ]:
sleep 5
kubectl logs -n $NRP_NAMESPACE -l app=jupyterhub,component=hub --tail=50


In [ ]:
kubectl get pods -n $NRP_NAMESPACE -l app=jupyterhub,component=singleuser-server


Troubleshooting is the standard Kubernetes trio: `describe` the failing pod, read namespace `events`, check hub/proxy `logs`.


## 7. Building custom images in NRP GitLab

The stock Jupyter images only go so far — real courses need their own package stacks. NRP GitLab ([gitlab.nrp-nautilus.io](https://gitlab.nrp-nautilus.io)) builds images for you in CI and hosts them in its container registry.

The workflow:

1. **Create a project** on NRP GitLab and add a `Dockerfile` — typically `FROM quay.io/jupyter/scipy-notebook:…` plus your `pip`/`conda` installs.
2. **Add `.gitlab-ci.yml`** — a single Kaniko job builds and pushes on every commit:

```yaml
image: ghcr.io/osscontainertools/kaniko:debug

stages:
- build-and-push

build-and-push-job:
  stage: build-and-push
  variables:
    GODEBUG: "http2client=0"
  script:
  - echo "{\"auths\":{\"$CI_REGISTRY\":{\"username\":\"$CI_REGISTRY_USER\",\"password\":\"$CI_REGISTRY_PASSWORD\"}}}" > /kaniko/.docker/config.json
  - /kaniko/executor --cache=true --push-retry=10 --context $CI_PROJECT_DIR --dockerfile $CI_PROJECT_DIR/Dockerfile --destination $CI_REGISTRY_IMAGE:$CI_COMMIT_SHORT_SHA --destination $CI_REGISTRY_IMAGE:latest
```


3. **Use the image** anywhere on the cluster — in a pod spec, or as a hub profile:

```yaml
  - display_name: My Course Image
    kubespawner_override:
      image_spec: gitlab-registry.nrp-nautilus.io/<group>/<project>:latest
```


Best practices: tag with commit SHAs (not just `latest`) so a course mid-semester never changes under your students; use `--cache=true` for fast rebuilds; keep credentials in CI variables, never in the Dockerfile.


## 8. End of tutorial — cleanup

Uninstall your Helm release so the cluster is left clean:


In [ ]:
helm uninstall $NRP_RELEASE -n $NRP_NAMESPACE


User PVCs are kept by default; delete them only if you're sure:


In [ ]:
kubectl delete pvc -n $NRP_NAMESPACE -l app=jupyterhub,component=singleuser-storage


### 🧠 Quick check — the capstone

**What role does the Helm values file play in your deployment?**

- It customizes the chart's templates — auth, images, storage, resources — in one YAML file
- It replaces kubectl for managing the cluster
- It builds the container images the hub uses

<details><summary><b>Reveal answer</b></summary>

**✔ It customizes the chart's templates — auth, images, storage, resources — in one YAML file**

The z2jh chart contains the templates for every hub/proxy/spawner object; your values file is the *entire* description of your deployment. Version-control it and you can rebuild the hub anywhere.

</details>

**Your course hub goes to production. What happens to the Dummy authenticator?**

- Swap it for CILogon/OIDC so students use campus credentials
- Keep it and share the password with the class
- Remove authentication entirely — the ingress is already HTTPS

<details><summary><b>Reveal answer</b></summary>

**✔ Swap it for CILogon/OIDC so students use campus credentials**

Dummy auth is a tutorial convenience. The workspace's `cilogon-jupyterhub-config.yaml` shows the production pattern: institutional login, allowlists, no passwords to distribute.

</details>

**Why tag course images with commit SHAs instead of only `latest`?**

- So the environment never changes underneath students mid-semester
- Because `latest` images pull more slowly
- Because GitLab requires unique tags

<details><summary><b>Reveal answer</b></summary>

**✔ So the environment never changes underneath students mid-semester**

`latest` moves every time CI runs. Pinning profiles to a SHA (or release tag) means the same image all semester — reproducibility is the whole reason you built a custom image.

</details>

**You edited `jhub-values.yaml` to add an ingress. How do the changes reach your running hub?**

- `helm upgrade <release> jupyterhub/jupyterhub --values yamls/jhub-values.yaml`
- `kubectl apply -f yamls/jhub-values.yaml`
- Delete the release and reinstall from scratch

<details><summary><b>Reveal answer</b></summary>

**✔ `helm upgrade <release> jupyterhub/jupyterhub --values yamls/jhub-values.yaml`**

A values file is chart *input*, not a Kubernetes manifest — `kubectl apply` on it fails. `helm upgrade` re-renders the templates with your new values and rolls out only what changed; you did this live when adding the ingress and profiles.

</details>

**Every student's server shows the same `/home/shared` folder. What makes that work?**

- One RWX CephFS PVC mounted into every user pod via `extraVolumes`/`extraVolumeMounts`
- Each student's home PVC is cloned from a master copy
- The hub copies the files into each home directory at spawn

<details><summary><b>Reveal answer</b></summary>

**✔ One RWX CephFS PVC mounted into every user pod via `extraVolumes`/`extraVolumeMounts`**

It's the `jupyterhub-shared-volume` claim from the storage episode — RWX means all user pods mount it simultaneously. Instructors drop a dataset in once; the whole class sees it instantly (mount it read-only for students in production).

</details>


## Where to go next

If you followed along on the training hub, that access is temporary — get your own so you can keep building.

**1. Register your identity.** NRP authenticates through **CILogon**, so you sign in with your existing campus account — no new password.

- Go to **[portal.nrp.ai](https://portal.nrp.ai)** and log in with CILogon (pick your institution).
- Follow [Getting started](https://nrp.ai/documentation/userdocs/start/getting-started/) to complete your profile.

**2. Get a namespace for your course.**

- **Joining an existing project?** Ask its admin to add you — send them the identity shown in the portal.
- **Starting a course?** Request a namespace and allocation via **[nrp.ai/contact](https://nrp.ai/contact/)**, which is also the Matrix channel used for live help. Say what you're teaching, how many students, and whether the course needs GPUs.

**3. Point `kubectl` at NRP.** Grab your kubeconfig from the portal ([get-config](https://nrp.ai/documentation/userdocs/start/getting-started/#cluster-access-via-kubectl)), drop it at `~/.kube/config`, and verify with `kubectl get pods -n <your-namespace>`. Everything in this notebook then works against your own course.

**4. Keep the materials.** Lessons and runnable notebooks stay online at **[training.nrp-nautilus.io](https://training.nrp-nautilus.io/)** and on **[GitHub](https://github.com/nrp-nautilus/nrp-training)**. Docs: **[nrp.ai/documentation](https://nrp.ai/documentation/)**.


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.


In [ ]:
bash check.sh 2
